# HRAI API demo notebook

In [ ]:
!pip install --upgrade pip
!pip install -r requirements.txt

!git lfs pull

In [25]:
import os, json, random, textwrap
from pathlib import Path
import requests

from config import conf
from load import get_encoder
from query import query_type, extract_from_resume
from suggestions import get_expanded_skills, get_domain_reports
from pos_extraction import text_to_ngrams

from dotenv import load_dotenv
import faiss
from rich import print

BASE_URL = 'http://127.0.0.1:8000'

## konfigurace modelu a metadat pro lokální volání
\*pro rychlejší fungování encoderu doporučuji nastavit HF_TOKEN v .env


In [2]:
load_dotenv()
db = faiss.read_index(os.path.join(conf.db_dir, f"all.index"))
with open(os.path.join(conf.data_dir, f"key_to_ent.json"),'r') as f:
    metadata = json.loads(f.read())
model = get_encoder()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## převedení textu na ngramy

In [10]:
resumes = json.loads(Path(os.path.join('testfiles', 'cs_resumes.json')).read_text(encoding='utf-8'))
text = random.choice(resumes)
print(text)

Šéfkuchař s 12 lety zkušeností s vařením v prostředí restaurací s vysokým tempem. Předchozí práce jako přípravný 
kuchař, šéfkuchař a zástupce šéfkuchaře. Energický kulinářský profesionál se směsí kreativity, vášně pro jídlo a 
výjimečných kuchařských dovedností. Funguje dobře jako dynamický vůdce v nastavení vysokého tlaku. Přednosti 
Soustředěná a disciplinovaná Schopnost velkoobjemové výroby Dobře vyladěná paleta Zaměření na kontrolu porcí a 
nákladů Znalost řízení zásob Dvojjazyčný (angličtina/) Zkušenosti Line Chef/Expediter až Současný název společnosti
Město, Stát Vedené směny při osobní přípravě potravin a provádění požadavků na základě požadovaných specifikací. 
Udržoval hladký a včasný provoz při přípravě a rozvozu jídel a sanitaci kuchyní. Řádně označeny a skladovány 
všechny suroviny včetně produktů, masa, ryb, drůbeže, mléčných výrobků a suchého zboží ve vhodné skladovací 
místnosti, průchozí lednici, mrazničce nebo chladicím boxu. Vyměnil a dezinfikoval všechna prkénka na krájení, 
lavice a povrchy při zahájení nového úkolu, aby se zabránilo křížové kontaminaci. Pravidelně komunikujeme s hosty, 
abychom získali zpětnou vazbu o kvalitě produktů a úrovni služeb. Připravovaná jídla důsledně a v souladu s 
recepty, pokyny pro porcování, vaření a kontrolu odpadu. Manažer kuchyně  až  Název společnosti Město, Stát 
Udržoval kvalifikovaný kuchyňský personál řádným koučováním, poradenstvím a disciplinací zaměstnanců. Poučil nové 
zaměstnance o správné přípravě jídla, skladování potravin, používání kuchyňského vybavení a náčiní, hygienických a 
bezpečnostních otázkách. Line Chef  Název společnosti Město, Stát Výrobky z hotových potravin důsledně a v souladu 
s recepty, pokyny pro porcování, vaření a kontrolu odpadu. Udržoval kvalifikovaný personál kuchyně řádným 
koučováním, poradenstvím a disciplinací zaměstnanců. Line Chef Název společnosti Město, Stát Připravené potraviny 
důsledně a v souladu s recepty, porcováním, vařením a pokyny pro kontrolu odpadu. Snížení nákladů na krmivo o 2 % 
procent použitím sezónních surovin, stanovením standardů pro velikost porcí a minimalizací odpadu. Udržoval si 
aktuální znalosti o místní konkurenci a trendech v restauračním průmyslu. Udržoval hladký a včasný provoz při 
přípravě a rozvozu jídel a sanitaci kuchyní. Line Chef až Název společnosti Město, Stát Připravené potraviny 
důsledně a v souladu s recepty, porcováním, vařením a pokyny pro kontrolu odpadu. Snížili jsme náklady na jídlo o 
10 % díky použití sezónních surovin, stanovení standardů pro velikost porcí a minimalizaci odpadu. Řídil personál 
kuchyně náborem, výběrem, najímáním, orientací, školením, přidělováním, plánováním, dohledem, hodnocením a 
vynucováním disciplíny v případě potřeby. Ověřil správnou velikost porcí a důsledně dosáhl vysokých standardů 
kvality potravin. Udržoval kvalifikovaný personál kuchyně řádným koučováním, poradenstvím a disciplinací 
zaměstnanců. Vytvářel menu, ceny a speciální nabídky jídel pro zvýšení tržeb a spokojenosti zákazníků. Kontrolovali
kuchyně za účelem sledování kvality přípravy jídel a servisu, vzhledu jídel a čistoty výrobních a servisních 
prostor. Vždy praktikoval bezpečné postupy při manipulaci s potravinami. Line Chef  až Název společnosti Město, 
Stát Led se směny při osobní přípravě jídel a vyřizování požadavků na základě požadovaných specifikací. Udržoval 
hladký a včasný provoz při přípravě a rozvozu jídel a sanitaci kuchyní. Řádně označeny a skladovány všechny 
suroviny včetně produktů, masa, ryb, drůbeže, mléčných výrobků a suchého zboží ve vhodné skladovací místnosti, 
průchozí lednici, mrazničce nebo chladicím boxu. Vyměnil a dezinfikoval všechna prkénka na krájení, lavice a 
povrchy při zahájení nového úkolu, aby se zabránilo křížové kontaminaci. Pravidelně komunikujeme s hosty, abychom 
získali zpětnou vazbu o kvalitě produktů a úrovni služeb. Připravovaná jídla důsledně a v souladu s recepty, pokyny
pro porcování, vaření a kontrolu odpadu. Linka Šéfkuchař / Myčka nád

In [11]:
ngrams = text_to_ngrams(text)
print(', '.join(sorted(ngrams)))

10 díky použití, 12 lety, 12 lety zkušeností, 2 procent, 2 procent použitím, aktuální znalosti, angličtina, 
angličtina jiný jazyk, ave, bezpečnost, bezpečnost plánování, bezpečnost plánování dohled, bezpečnostních otázkách,
bezpečné postupy, boxu, cenotvorba, cenotvorba kvalita, cenotvorba kvalita nábor, ceny, certifikát, certifikát 
dovednosti, certifikát dovednosti koučování, chef až název, chef název, chef název společnosti, chladicím boxu, 
city stát, college city, college city stát, community college city, dezinfikoval všechna prkénka, diplom east 
jefferson, disciplinací, disciplinací zaměstnanců, disciplinovaná schopnost, disciplíny, disciplíny v případě, 
dobře vyladěná paleta, dodávka, dodávka nábor, dodávka nábor cenotvorba, dohled, dohledem, dohledem hodnocením, 
dosáhl vysokých standardů, dovednosti, dovednosti koučování, dovednosti koučování vaření, dovedností, drůbeže, 
drůbeže mléčných výrobků, dvojjazyčný angličtina, dynamický vůdce, díky použití, east jefferson, east jefferson 
high, energický kulinářský profesionál, harahan la linka, hickory ave, hodnocením, hodnocením a vynucováním, hosty,
hosty aby bychom, hotových potravin, hotových potravin důsledně, jazyk, jazyk zkušenosti, jazyk zkušenosti line, 
jiný jazyk, jiný jazyk zkušenosti, jídel, jídel a sanitaci, jídel a servisu, jídel a vyřizování, jídel a čistoty, 
jídel pro zvýšení, jídla, jídla důsledně, jídla skladování, jídla skladování potravin, jídlo, jídlo o 10, 
komunikujeme s hosty, konkurenci, konkurenci a trendech, kontaminaci, kontrolovali kuchyně, kontrolu, kontrolu 
odpadu, kontrolu porcí, koučování, koučování vaření, koučování vaření poradenství, koučováním, koučováním 
poradenstvím, kreativity, kreativity vášně, krmivo, krmivo o 2, krájení, krájení lavice, kuchař, kuchař myčka, 
kuchař myčka nádobí, kuchař šéfkuchař, kuchařských dovedností, kuchyní, kuchyně, kuchyně až název, kuchyně náborem,
kuchyně náborem výběrem, kuchyně za účelem, kuchyně řádným koučováním, kuchyňského vybavení, kuchyňský personál, 
kulinářské umění, kulinářské umění certifikát, kulinářský profesionál, kurz v pohostinství, kvalifikovaný kuchyňský
personál, kvalifikovaný personál, kvalifikovaný personál kuchyně, kvalita, kvalita nábor, kvalita nábor bezpečnost,
kvality, kvality potravin, kvality přípravy, kvality přípravy jídel, kvalitě, kvalitě produktů, křížové, křížové 
kontaminaci, la linka, la linka kuchař, lavice, lavice a povrchy, led se směny, lednici, lednici mrazničce, lety, 
lety zkušeností, line chef název, linka, linka kuchař, linka kuchař myčka, linka šéfkuchař, management, management 
delgado, managementu, managementu houston, managementu houston university, manažer, manažer kuchyně, manipulaci, 
manipulaci s potravinami, masa, masa ryb, masa ryb drůbeže, menu, menu ceny, minimalizaci, minimalizaci odpadu, 
minimalizací, minimalizací odpadu, mléčných výrobků, mrazničce, myčka, myčka nádobí, místnosti, místnosti průchozí 
lednici, místní konkurenci, město, město stát, město stát udržoval, město stát verona, město stát výrobky, nabídky,
nabídky jídel, najímáním, najímáním orientací, najímáním orientací školením, nastavení, nastavení vysokého tlaku, 
nového úkolu, nábor, nábor bezpečnost, nábor bezpečnost plánování, nábor cenotvorba, nábor cenotvorba kvalita, 
náborem, náborem výběrem, náborem výběrem najímáním, nádobí, náklady, náklady na jídlo, nákladů, nákladů na krmivo,
nákladů znalost, nákladů znalost řízení, název, název společnosti, název společnosti město, náčiní, odpadu, 
orientací, orientací školením, orientací školením přidělováním, osobní přípravě, osobní přípravě jídel, osobní 
přípravě potravin, otázkách, ověřil správnou velikost, paleta, paleta zaměření, personál, personál kuchyně, 
personál kuchyně náborem, personál řádným koučováním, plánování, plánování dohled, plánováním, plánováním dohledem,
plánováním dohledem hodnocením, pohostinství, pohostinství management, pohostinství management delgado, pokyny, 
pokyny pro kontrolu, pokyny pro porcov

## API endpointy: /resume/skills a /resume/domains (PDF, DOCX, ODT)

In [20]:
def post_file(endpoint: str, filename: str):
    file_path = Path('testfiles') / filename
    files = {'file': (file_path.name, file_path.read_bytes(), 'application/octet-stream')}
    resp = requests.post(f"{BASE_URL}{endpoint}", files=files)
    print({'endpoint': endpoint, 'file': filename, 'status': resp.status_code, 'json': resp.json()})

#post_file('/resume/skills', 'Životopis.pdf')
post_file('/resume/domains', 'resume.pdf')

{
    'endpoint': '/resume/domains',
    'file': 'resume.pdf',
    'status': 200,
    'json': [
        {
            'domain': 'Specialisté',
            'occ_count': 26,
            'occupations': [
                {
                    'occupation': {
                        'id': '8063',
                        'cosine_score': 0.862810492515564,
                        'esco_uri': 'http://data.europa.eu/esco/occupation/b89ba3a8-f03a-46b8-ac54-c87d7204aad0',
                        'label': 'mluvčí',
                        'code': '2432.9.1',
                        'description': 'Mluvčí hovoří jménem společností nebo organizací. Využívají komunikační 
strategie k zastupování klientů prostřednictvím veřejných oznámení a konferencí. Vykreslují své klienty v 
pozitivním světle a snaží se blíže vysvětlit jejich zájmy a činnost.'
                    },
                    'missing_skills': [
                        {
                            'id': '16598',
                            'esco_uri': 'http://data.europa.eu/esco/skill/5753e2ca-8934-45d3-8e52-3877d373239d',
                            'label': 'diplomatické zásady',
                            'relation_type': 'essential',
                            'description': 'Postupy přispívající k uzavírání dohod nebo mezinárodních smluv s 
jinými zeměmi prostřednictvím vedení jednání a vynakládání úsilí na ochranu zájmů domácí vlády, jakož i postupy 
usilující o vyjednání kompromisu.'
                        },
                        {
                            'id': '27917',
                            'esco_uri': 'http://data.europa.eu/esco/isced-f/00',
                            'label': 'všeobecné programy a kvalifikace',
                            'relation_type': 'optional',
                            'description': ''
                        },
                        {
                            'id': '20280',
                            'esco_uri': 'http://data.europa.eu/esco/skill/9194696f-8a5a-4f7b-bf8a-a680da41c320',
                            'label': 'utváření veřejného mínění',
                            'relation_type': 'essential',
                            'description': 'Proces, v jehož rámci se formulují a prosazují názory a stanoviska ve 
vztahu k něčemu. Prvky, které hrají roli ve veřejném mínění, jako je formování informací, psychické procesy a 
stádní chování.'
                        },
                        {
                            'id': '21798',
                            'esco_uri': 'http://data.europa.eu/esco/skill/aa56a80c-b4d5-495c-9c15-70b5a7933911',
                            'label': 'rétorika',
                            'relation_type': 'essential',
                            'description': 'Umění projevu, které si klade za cíl zlepšit schopnost autorů a řečníků
informovat, přesvědčovat nebo motivovat své publikum.'
                        },
                        {
                            'id': '27179',
                            'esco_uri': 'http://data.europa.eu/esco/skill/fe30a4b0-1a99-4fc7-afff-b162b8c834ba',
                            'label': 'komunikační zásady',
                            'relation_type': 'essential',
                            'description': 'Soubor společně sdílených zásad, pokud jde o komunikaci, jako je 
aktivní naslouchání, vytvoření harmonického vztahu, přizpůsobení vyjadřování a respektování zásahů jiných.'
                        },
                        {
                            'id': '12582',
                            'esco_uri': 'http://data.europa.eu/esco/skill/1ba59ce0-7fec-434b-8d5c-9b275250a26c',
                            'label': 'připravovat prezentační materiály',
                            'relation_type': 'essential',
                            'description': 'Připravovat dokumenty, prezentace, plakáty a další média potřebná pro 
specifické publikum.'
                        },
                        {
                            'id': '13240',
 

## API endpointy: /text/skills a /text/domains

In [16]:
text_req = {
    'occupations': ['software developer', 'data analyst'],
    'skills': ['python', 'strojové učení', 'sql databáze']
}
skills_resp = requests.post(f"{BASE_URL}/text/skills", json=text_req)
print({'endpoint': '/text/skills', 'status': skills_resp.status_code, 'json': skills_resp.json()[0]})

domains_resp = requests.post(f"{BASE_URL}/text/domains", json=text_req)
print({'endpoint': '/text/domains', 'status': domains_resp.status_code, 'json': domains_resp.json()[0]})

{
    'endpoint': '/text/skills',
    'status': 200,
    'json': {
        'occupation': {
            'id': '10307',
            'cosine_score': 0.8669745922088623,
            'esco_uri': 'http://data.europa.eu/esco/occupation/f2b15a0e-e65a-438a-affb-29b9d50b77d1',
            'label': 'softwarový vývojář/softwarová vývojářka / programátor softwaru',
            'code': '2512.4',
            'description': 'Vývojáři softwaru implementují nebo programují všechny typy softwarových systémů 
založených na specifikacích a návrzích s použitím programovacích jazyků, nástrojů a platforem.'
        },
        'missing_skills': [
            {
                'id': '12887',
                'esco_uri': 'http://data.europa.eu/esco/skill/209a5498-3449-4689-8ed9-bd08cab4fd78',
                'label': 'technické zásady',
                'relation_type': 'essential',
                'description': 'Technická a strojírenská hlediska, jako je funkčnost, reprodukovatelnost a náklady 
související s koncepcí či návrhem, a způsob, jakým jsou uplatňovány při dokončování inženýrských projektů.'
            },
            {
                'id': '27917',
                'esco_uri': 'http://data.europa.eu/esco/isced-f/00',
                'label': 'všeobecné programy a kvalifikace',
                'relation_type': 'optional',
                'description': ''
            },
            {
                'id': '18194',
                'esco_uri': 'http://data.europa.eu/esco/skill/7111b95d-0ce3-441a-9d92-4c75d05c4388',
                'label': 'projektový management',
                'relation_type': 'essential',
                'description': 'Porozumění řízení projektů a činnostem, které jsou součástí této oblasti. Znalost 
různých aspektů řízení projektů, jako jsou čas, zdroje, požadavky, lhůty a reakce na neočekávané události.'
            },
            {
                'id': '18306',
                'esco_uri': 'http://data.europa.eu/esco/skill/72a74f69-5cf1-43c5-99b9-62a444578919',
                'label': 'konstrukční procesy',
                'relation_type': 'essential',
                'description': 'Systematický přístup k vývoji a údržbě inženýrských systémů.'
            },
            {
                'id': '11094',
                'esco_uri': 'http://data.europa.eu/esco/skill/04dfd9fb-e0cf-40f6-96c6-9d2280c4347e',
                'label': 'poskytovat technickou dokumentaci',
                'relation_type': 'essential',
                'description': 'Připravit dokumentaci pro stávající a budoucí výrobky nebo služby popisující jejich
funkčnost a složení tak, aby byla srozumitelná široké veřejnosti bez technických vědomostí a byla v souladu s 
definovanými požadavky a normami. Průběžně aktualizovat dokumentaci.'
            },
            {
                'id': '11737',
                'esco_uri': 'http://data.europa.eu/esco/skill/0e0c9c11-b39e-43cf-ba95-2fe646f5de64',
                'label': 'identifikovat požadavky zákazníků',
                'relation_type': 'essential',
                'description': 'Používat techniky a nástroje, jako jsou průzkumy, dotazníky a aplikace ICT, ke 
zjištění, definování, analýze, zaznamenání a uchovávání uživatelských požadavků na systém, službu nebo výrobek.'
            },
            {
                'id': '17783',
                'esco_uri': 'http://data.europa.eu/esco/skill/6a899482-8bcf-40ca-ae67-579a1cc467ab',
                'label': 'řídit inženýrský projekt',
                'relation_type': 'essential',
                'description': 'Řídit zdroje, rozpočet, lhůty a lidské zdroje inženýrského projektu a plánovat 
harmonogramy a veškeré technické činnosti související s projektem.'
            },
            {
                'id': '20156',
                'esco_uri': 'http://data.europa.eu/esco/skill/901105f8-3a08-453e-b1dc-23619e317fd3',
                'label': 'vytvářet vývojové diagramy',
                'relation_type': 'essential',
                'description': 'Vytvořit diagram,

{
    'endpoint': '/text/domains',
    'status': 200,
    'json': {
        'domain': 'Specialisté',
        'occ_count': 2,
        'occupations': [
            {
                'occupation': {
                    'id': '10307',
                    'cosine_score': 0.8669745922088623,
                    'esco_uri': 'http://data.europa.eu/esco/occupation/f2b15a0e-e65a-438a-affb-29b9d50b77d1',
                    'label': 'softwarový vývojář/softwarová vývojářka / programátor softwaru',
                    'code': '2512.4',
                    'description': 'Vývojáři softwaru implementují nebo programují všechny typy softwarových 
systémů založených na specifikacích a návrzích s použitím programovacích jazyků, nástrojů a platforem.'
                },
                'missing_skills': [
                    {
                        'id': '12887',
                        'esco_uri': 'http://data.europa.eu/esco/skill/209a5498-3449-4689-8ed9-bd08cab4fd78',
                        'label': 'technické zásady',
                        'relation_type': 'essential',
                        'description': 'Technická a strojírenská hlediska, jako je funkčnost, reprodukovatelnost a 
náklady související s koncepcí či návrhem, a způsob, jakým jsou uplatňovány při dokončování inženýrských projektů.'
                    },
                    {
                        'id': '27917',
                        'esco_uri': 'http://data.europa.eu/esco/isced-f/00',
                        'label': 'všeobecné programy a kvalifikace',
                        'relation_type': 'optional',
                        'description': ''
                    },
                    {
                        'id': '18194',
                        'esco_uri': 'http://data.europa.eu/esco/skill/7111b95d-0ce3-441a-9d92-4c75d05c4388',
                        'label': 'projektový management',
                        'relation_type': 'essential',
                        'description': 'Porozumění řízení projektů a činnostem, které jsou součástí této oblasti. 
Znalost různých aspektů řízení projektů, jako jsou čas, zdroje, požadavky, lhůty a reakce na neočekávané události.'
                    },
                    {
                        'id': '18306',
                        'esco_uri': 'http://data.europa.eu/esco/skill/72a74f69-5cf1-43c5-99b9-62a444578919',
                        'label': 'konstrukční procesy',
                        'relation_type': 'essential',
                        'description': 'Systematický přístup k vývoji a údržbě inženýrských systémů.'
                    },
                    {
                        'id': '11094',
                        'esco_uri': 'http://data.europa.eu/esco/skill/04dfd9fb-e0cf-40f6-96c6-9d2280c4347e',
                        'label': 'poskytovat technickou dokumentaci',
                        'relation_type': 'essential',
                        'description': 'Připravit dokumentaci pro stávající a budoucí výrobky nebo služby 
popisující jejich funkčnost a složení tak, aby byla srozumitelná široké veřejnosti bez technických vědomostí a byla
v souladu s definovanými požadavky a normami. Průběžně aktualizovat dokumentaci.'
                    },
                    {
                        'id': '11737',
                        'esco_uri': 'http://data.europa.eu/esco/skill/0e0c9c11-b39e-43cf-ba95-2fe646f5de64',
                        'label': 'identifikovat požadavky zákazníků',
                        'relation_type': 'essential',
                        'description': 'Používat techniky a nástroje, jako jsou průzkumy, dotazníky a aplikace ICT,
ke zjištění, definování, analýze, zaznamenání a uchovávání uživatelských požadavků na systém, službu nebo výrobek.'
                    },
                    {
                        'id': '17783',
                        'esco_uri': 'http://data.europa.eu/esco/skill/6a899482-8bcf-40ca-ae67-579a1cc467ab',
                        'label': 'řídit inženýrský projekt',
 

## API endpoint: /query (každý typ dotazu)

In [21]:
query_cases = [
    {'query': 'finančník', 'query_type': 'occupation'},
    {'query': 'napomínání lidí', 'query_type': 'skill', 'min_set_score':0.5},
    {'query': 'řemesla', 'query_type': 'isco_group'},
    {'query': 'správa sítí', 'query_type': 'skill_group'},
]
for case in query_cases:
    resp = requests.post(f"{BASE_URL}/query", json=case)
    print({'endpoint': '/query', 'request': case, 'status': resp.status_code, 'json': resp.json()})

{
    'endpoint': '/query',
    'request': {'query': 'finančník', 'query_type': 'occupation'},
    'status': 200,
    'json': [
        {
            'id': '8562',
            'cosine_score': 0.6521406173706055,
            'entity_type': 'occupation',
            'esco_uri': 'http://data.europa.eu/esco/occupation/c3aae59e-1441-4b3e-bdd1-6da50dc7cf42',
            'label': 'účetní',
            'code': '3313.2',
            'description': 'Účetní zaznamenávají a sestavují každodenní finanční transakce organizace nebo 
společnosti, obvykle jde o prodej, nákupy, platby a příjmy. Zajišťují, aby všechny finanční transakce byly 
zdokumentovány v příslušném (denním) portfoliu a hlavní účetní knize, a aby byly vyváženy. Účetní připravují zápisy
a účetní knihy s finančními transakcemi pro účetního revizora k analýze bilancí a výsledovky.'
        }
    ]
}

{
    'endpoint': '/query',
    'request': {'query': 'napomínání lidí', 'query_type': 'skill', 'min_set_score': 0.5},
    'status': 200,
    'json': [
        {
            'id': '24571',
            'cosine_score': 0.63362056016922,
            'entity_type': 'skill',
            'esco_uri': 'http://data.europa.eu/esco/skill/d6a58927-b8e0-4863-b4e0-de0e63cf7661',
            'label': 'získat si pozornost lidí',
            'code': '',
            'description': 'Přistupovat k lidem a upozorňovat je na téma, které je jim prezentováno, nebo od nich 
získávat informace.'
        },
        {
            'id': '13870',
            'cosine_score': 0.6248283982276917,
            'entity_type': 'skill',
            'esco_uri': 'http://data.europa.eu/esco/skill/2e7a8114-3e1b-4ca5-98a4-e0dc5abfe827',
            'label': 'interpretovat vystupování lidí',
            'code': '',
            'description': 'Shromažďovat informace o osobách prostřednictvím pečlivého pozorování jejich řeči těla,
zaznamenávání zvukových podnětů a kladení otázek.'
        },
        {
            'id': '19539',
            'cosine_score': 0.593048095703125,
            'entity_type': 'skill',
            'esco_uri': 'http://data.europa.eu/esco/skill/8657a613-7567-44d5-89aa-36550c5c0ac9',
            'label': 'vypátrat osoby',
            'code': '',
            'description': 'Zjistit místo pobytu osob, které jsou pohřešovány nebo si nepřejí být nalezeny.'
        },
        {
            'id': '26806',
            'cosine_score': 0.5840922594070435,
            'entity_type': 'skill',
            'esco_uri': 'http://data.europa.eu/esco/skill/f8e5f1fd-82ee-45c0-b179-399d8e9a22aa',
            'label': 'jednat s problematickými lidmi',
            'code': '',
            'description': 'Zajišťovat bezpečnou práce a účinnou komunikaci s jednotlivci a skupinami osob, které 
se nacházejí v obtížné situaci, což zahrnuje rozpoznání známek agrese, rozrušení, vyhrožování a způsobu jejich 
řešení s cílem podpořit osobní bezpečnost a bezpečnost ostatních.'
        },
        {
            'id': '21798',
            'cosine_score': 0.5828225612640381,
            'entity_type': 'skill',
            'esco_uri': 'http://data.europa.eu/esco/skill/aa56a80c-b4d5-495c-9c15-70b5a7933911',
            'label': 'rétorika',
            'code': '',
            'description': 'Umění projevu, které si klade za cíl zlepšit schopnost autorů a řečníků informovat, 
přesvědčovat nebo motivovat své publikum.'
        },
        {
            'id': '24099',
            'cosine_score': 0.5816469192504883,
            'entity_type': 'skill',
            'esco_uri': 'http://data.europa.eu/esco/skill/ceb13e3a-835e-47e3-a1c9-34dfa21de0fd',
            'label': 'poučit veřejnost',
            'code': '',
            'description': 'Dávat pokyny veřejnosti v situacích, kdy se chovají způsobem, který není v souladu s 
právními předpisy, nebo je řídit při mimořádných situacích.'
        },
        {
            'id': '11067',
            'cosine_score': 0.5711115002632141,
            'entity_type': 'skill',
            'esco_uri': 'http://data.europa.eu/esco/skill/045f71e6-0699-4169-8a54-9c6b96f3174d',
            'label': 'radit ostatním',
            'code': '',
            'description': 'Nabízet návrhy týkající se nejlepšího doporučeného postupu.'
        },
        {
            'id': '20454',
            'cosine_score': 0.5684691071510315,
            'entity_type': 'skill',
            'esco_uri': 'http://data.europa.eu/esco/skill/94c47c75-815b-403f-990c-1210e950e0b9',
            'label': 'vyslýchat',
            'code': '',
            'description': 'Vyslýchat osoby tak, aby poskytly informace, které by mohly být užitečné pro 
vyšetřování a které by se případně mohly pokusit zatajit.'
        }
    ]
}

{
    'endpoint': '/query',
    'request': {'query': 'řemesla', 'query_type': 'isco_group'},
    'status': 200,
    'json': [
        {
            'id': '14347',
            'cosine_score': 0.8401617407798767,
            'entity_type': 'skill',
            'esco_uri': 'http://data.europa.eu/esco/skill/36065f25-3bbe-4a5c-a0fe-5623ad4b20df',
            'label': 'řemeslné zpracování',
            'code': '',
            'description': 'Techniky používané k navrhování ručně vyráběného zboží.'
        },
        {
            'id': '14433',
            'cosine_score': 0.7076001167297363,
            'entity_type': 'skill',
            'esco_uri': 'http://data.europa.eu/esco/skill/3782a625-52c2-46c7-b585-35ba4ddc4a1d',
            'label': 'řemeslná dovednost',
            'code': '',
            'description': 'Schopnost pracovat rukama za účelem vytvoření určitého uměleckého díla.'
        },
        {
            'id': '11112',
            'cosine_score': 0.6892766356468201,
            'entity_type': 'skill',
            'esco_uri': 'http://data.europa.eu/esco/skill/05338a70-1ea1-4c33-a886-1c4f0c7d7241',
            'label': 'předávat řemeslné techniky',
            'code': '',
            'description': 'Předávat znalosti a dovednosti, vysvětlovat a ukazovat používání zařízení a materiálů a
odpovídat na otázky týkající se řemeslných technik při výrobě produktů.'
        },
        {
            'id': '27941',
            'cosine_score': 0.6657117605209351,
            'entity_type': 'skill_group',
            'esco_uri': 'http://data.europa.eu/esco/isced-f/021',
            'label': 'umění',
            'code': '',
            'description': ''
        },
        {
            'id': '12454',
            'cosine_score': 0.6553968787193298,
            'entity_type': 'skill',
            'esco_uri': 'http://data.europa.eu/esco/skill/19858dd3-a5fe-4855-b644-6ecfefd1c384',
            'label': 'tesařství',
            'code': '',
            'description': 'Stavební metody vztahující se k dřevěným prvkům, jako je stavba střech, podlah a budov 
s dřevěnou konstrukcí, jakož i dalších souvisejících výrobků, jako jsou dveře nebo sokly.'
        }
    ]
}

{
    'endpoint': '/query',
    'request': {'query': 'správa sítí', 'query_type': 'skill_group'},
    'status': 200,
    'json': [
        {
            'id': '28034',
            'cosine_score': 0.7260320782661438,
            'entity_type': 'skill_group',
            'esco_uri': 'http://data.europa.eu/esco/isced-f/0612',
            'label': 'vytváření a správa databází a sítí',
            'code': '',
            'description': ''
        }
    ]
}

## Přímé volání query_type pro všechny ent_type

In [22]:
entry = ['rybář']
ent_type = 'occupation'

results = query_type(db=db, metadata=metadata, model=model, ents=entry, ent_type=ent_type, min_score=0.5)
print(f"{entry} ({ent_type})")
print(results)

['rybář'] (occupation)

[
    EntityResult(
        id='10830',
        cosine_score=0.6806809306144714,
        entity_type='occupation',
        esco_uri='http://data.europa.eu/esco/occupation/ff63fa99-c634-4e6f-a693-6a49482952e9',
        label='kapitán rybářské lodi/kapitánka rybářské lodi',
        code='6223.2.1',
        description='Kapitáni rybářských lodí provozují rybářská plavidla v pobřežních vodách, přičemž provádějí 
činnost na palubě a v podpalubí. Řídí plavbu, jakož i lov a konzervaci ryb v rámci stanovených hranic v souladu s 
vnitrostátními a mezinárodními předpisy.'
    )
]

## Pipeline: extract_from_resume -> get_expanded_skills -> get_domain_reports

In [29]:
random_resume = random.choice(resumes)
print(textwrap.shorten(f'sample text: {random_resume}', width=600, placeholder="..."))

extracted_entities = extract_from_resume(db, metadata, model, random_resume_text)
print(textwrap.shorten(f'extracted entities: {extracted_entities}', width=600, placeholder="..."))

skill_suggestions = get_expanded_skills(metadata, extracted_entities)
print(textwrap.shorten(f'skill suggestions: {skill_suggestions}', width=600, placeholder="..."))

domain_reports = get_domain_reports(skill_suggestions)
print(textwrap.shorten(f'domain stats: {domain_reports}', width=600, placeholder="..."))

sample text: Zdravotní péče Administrativní asistent s 3 lety zkušeností v oblasti zdravotnictví po dobu 5+ let 
Přidělený měsíční rozpočet USD na plánované aktivity Asociace muslimských studentů Organizované fundraisingové akce
pod dohledem koordinátora událostí ve Wingově programu Dovednosti Zkušenosti s Microsoft Office Suite, Adobe 
Premier Suite, SQL databází a SAS Plánování a vývoj Strategické rozhodování Hovoří plynně anglicky a urdsky, 
Funkční ve španělštině Správa dat Vynikající komunikační dovednosti Time management Řešení konfliktů Pracovní 
historie Zdravotnictví Specialista na...

extracted entities: [EntityResult(id='15082', cosine_score=0.843226432800293, entity_type='skill', 
esco_uri='http://data.europa.eu/esco/skill/4134622c-c3fb-4a41-beb6-6d58ba5107db', label='prokázat odborné 
znalosti', code='', description='Prokázat rozsáhlé znalosti a komplexní porozumění určité oblasti výzkumu, včetně 
odpovědného výzkumu, etiky výzkumu a zásad vědecké integrity, požadavků ochrany soukromí a GDPR v souvislosti s 
výzkumnou činností v konkrétním oboru.'), EntityResult(id='4180', cosine_score=0.6944096088409424, 
entity_type='occupation',...

skill suggestions: [Suggestion(occupation=Occupation(id='4180', cosine_score=0.6944096088409424, 
esco_uri='http://data.europa.eu/esco/occupation/5e4d156a-1f05-4b2c-b0f0-5abb82d52ed6', label='klientský 
pracovník/klientská pracovnice / klientský pracovník', code='3312.1', description='Klientští pracovníci radí 
potenciálním klientům o druhu bankovních účtů vhodných pro jejich potřeby. Pracují s klienty a zřizují bankovní 
účet, jsou hlavním kontaktním místem v bance a rovněž pomáhají při veškeré nezbytné dokumentaci. Klienstští 
pracovníci mohou svým klientům doporučit, aby se obrátili na jiná...

domain stats: [DomainResult(domain='Obsluha strojů a zařízení, montéři', occ_count=6, 
occupations=[Suggestion(occupation=Occupation(id='27863', cosine_score=0.8986387252807617, 
esco_uri='http://data.europa.eu/esco/isco/C8344', label='Obsluha vysokozdvižných vozíků', code=8344, 
description=''), missing_skills=[]), Suggestion(occupation=Occupation(id='27863', cosine_score=0.8768764734268188, 
esco_uri='http://data.europa.eu/esco/isco/C8344', label='Obsluha vysokozdvižných vozíků', code=8344, 
description=''), missing_skills=[]), Suggestion(occupation=Occupation(id='27863',...